# Mini-DLSS Colab GPU Training

Clone this repo in Colab, open this notebook from the cloned checkout, then use `Runtime > Change runtime type > GPU` before running. The notebook follows the repo's next training steps: first run the fast-cycle `temporal_small` model, then optionally run the full-budget `temporal_full` model. Checkpoints and logs are written to Google Drive so the run can resume across Colab sessions.

## 1. GPU and Repository Setup

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import os

candidates = [
    Path.cwd(),
    Path("/content/Mini-DLSS"),
    Path("/content/mini-dlss"),
]
candidates.extend(path.parent for path in Path("/content").glob("*/train.py"))

for candidate in candidates:
    if (candidate / "train.py").exists() and (candidate / "configs/temporal_small.toml").exists():
        REPO_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError("Clone the repo into /content/Mini-DLSS, then reopen or rerun this notebook.")

os.chdir(REPO_DIR)
print("repo:", Path.cwd())

In [ ]:
!pip -q install -r requirements.txt kagglehub

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/mini-dlss")
DRIVE_RESULTS = DRIVE_ROOT / "results"
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

# train.py writes to ./results. Link that directory to Drive for durable checkpoints.
local_results = Path("results")
if local_results.exists() and not local_results.is_symlink():
    snapshot = Path("results_repo_snapshot")
    if not snapshot.exists():
        local_results.rename(snapshot)
    else:
        shutil.rmtree(local_results)
if not local_results.exists():
    os.symlink(DRIVE_RESULTS, local_results, target_is_directory=True)

print("Drive-backed results:", DRIVE_RESULTS)

## 2. Download and Prepare Vimeo-90K Union Data

This uses the two Kaggle subsets already assumed by the repo. If Kaggle asks for credentials, upload your `kaggle.json` to Colab and place it at `/root/.kaggle/kaggle.json` before rerunning this cell.

In [ ]:
import os
from pathlib import Path
import kagglehub
import subprocess

os.environ["KAGGLEHUB_CACHE"] = "/content/data/raw/kagglehub"

vimeo_a = Path(kagglehub.dataset_download("wangsally/vimeo-90k-3"))
vimeo_b = Path(kagglehub.dataset_download("wangsally/vimeo-90k-4"))

print("vimeo-90k-3:", vimeo_a)
print("vimeo-90k-4:", vimeo_b)

In [ ]:
subprocess.run(["python", "data/scripts/freeze_vimeo_union_splits.py", "--vimeo-a", str(vimeo_a), "--vimeo-b", str(vimeo_b)], check=True)
subprocess.run(["python", "data/scripts/build_vimeo_union_root.py", "--seq-a", str(vimeo_a / "sequence"), "--seq-b", str(vimeo_b / "sequence"), "--output", "data/raw/vimeo90k_union/sequence"], check=True)
subprocess.run(["python", "data/scripts/create_reds_val_from_vimeo.py", "--vimeo-root", "data/raw/vimeo90k_union/sequence", "--source-manifest", "data/splits/vimeo90k_union_val.txt", "--reds-ids", "data/splits/reds_val.txt", "--out-hr", "data/raw/reds/val_sharp", "--out-lr-x2", "data/raw/reds/val_sharp_bicubic/X2"], check=True)

In [ ]:
subprocess.run(["python", "data/scripts/check_dataset.py", "--hr-root", "data/raw/vimeo90k_union/sequence", "--manifest", "data/splits/vimeo90k_union_train.txt", "--num-frames", "5", "--scale", "2", "--generate-lr-on-the-fly", "--preview", "2"], check=True)
subprocess.run(["python", "data/scripts/check_dataset.py", "--hr-root", "data/raw/reds/val_sharp", "--lr-root", "data/raw/reds/val_sharp_bicubic/X2", "--manifest", "data/splits/reds_val.txt", "--num-frames", "5", "--scale", "2", "--preview", "2"], check=True)

## 3. Fast-Cycle Training: `temporal_small`

Start here. This verifies the full Colab GPU training/eval path without committing to a long run. Rerunning this cell resumes from `latest.pt` when present.

In [ ]:
from pathlib import Path
import json
import subprocess

TRAIN_HR = "data/raw/vimeo90k_union/sequence"
VAL_HR = "data/raw/reds/val_sharp"
VAL_LR = "data/raw/reds/val_sharp_bicubic/X2"

FAST_STEPS = 50000      # Use 2000 for a quick smoke run, 50000 for the planned fast cycle.
FAST_BATCH = 4          # Drop to 2 if a Colab T4 runs out of memory.
FAST_VAL_BATCHES = 20   # Use 0 for full validation; 20 is faster during iteration.

fast_override = {
    "dataset": {
        "train_hr_root": TRAIN_HR,
        "train_lr_root": "",
        "val_hr_root": VAL_HR,
        "val_lr_root": VAL_LR,
        "train_manifest": "data/splits/vimeo90k_union_train.txt",
        "val_manifest": "data/splits/reds_val.txt",
        "generate_lr_on_the_fly": True,
    },
    "train": {
        "max_steps": FAST_STEPS,
        "batch_size": FAST_BATCH,
        "num_workers": 2,
        "val_interval": 2000,
        "checkpoint_interval": 2000,
    },
    "eval": {"max_batches": FAST_VAL_BATCHES},
}

fast_resume = Path("results/runs/temporal_vsr_5f_small_2x/checkpoints/latest.pt")
if fast_resume.exists():
    fast_override["train"]["resume"] = str(fast_resume)

subprocess.run(
    ["python", "train.py", "--config", "configs/temporal_small.toml", "--override", json.dumps(fast_override)],
    check=True,
)

In [ ]:
subprocess.run(
    [
        "python", "eval.py",
        "--config", "configs/temporal_small.toml",
        "--override", json.dumps(fast_override),
        "--checkpoint", "results/runs/temporal_vsr_5f_small_2x/checkpoints/best.pt",
        "--save-images", "8",
        "--save-videos", "2",
    ],
    check=True,
)

## 4. Full Training: `temporal_full`

Run this after the fast cycle is healthy. This is the Week 5 quality-focused run and can span multiple Colab sessions. Rerunning resumes from `latest.pt` when present.

In [ ]:
FULL_STEPS = 300000
FULL_BATCH = 2
FULL_VAL_BATCHES = 0    # Full validation for reportable runs.

full_override = {
    "dataset": {
        "train_hr_root": TRAIN_HR,
        "train_lr_root": "",
        "val_hr_root": VAL_HR,
        "val_lr_root": VAL_LR,
        "train_manifest": "data/splits/vimeo90k_union_train.txt",
        "val_manifest": "data/splits/reds_val.txt",
        "generate_lr_on_the_fly": True,
    },
    "train": {
        "max_steps": FULL_STEPS,
        "batch_size": FULL_BATCH,
        "num_workers": 2,
        "val_interval": 5000,
        "checkpoint_interval": 5000,
    },
    "eval": {"max_batches": FULL_VAL_BATCHES},
}

full_resume = Path("results/runs/temporal_vsr_5f_full_2x/checkpoints/latest.pt")
if full_resume.exists():
    full_override["train"]["resume"] = str(full_resume)

subprocess.run(
    ["python", "train.py", "--config", "configs/temporal_full.toml", "--override", json.dumps(full_override)],
    check=True,
)

In [ ]:
subprocess.run(
    [
        "python", "eval.py",
        "--config", "configs/temporal_full.toml",
        "--override", json.dumps(full_override),
        "--checkpoint", "results/runs/temporal_vsr_5f_full_2x/checkpoints/best.pt",
        "--save-images", "8",
        "--save-videos", "2",
    ],
    check=True,
)

## 5. Inspect Artifacts

In [ ]:
from pathlib import Path

patterns = [
    "results/runs/*/checkpoints/best.pt",
    "results/runs/*/checkpoints/latest.pt",
    "results/runs/*/train_log.jsonl",
    "results/runs/*/val_metrics.jsonl",
    "results/tables/*eval.*",
    "results/videos/**/*.png",
    "results/videos/**/*.mp4",
]

for pattern in patterns:
    matches = sorted(Path().glob(pattern))[:20]
    if matches:
        print(f"\n{pattern}")
        for path in matches:
            print(" ", path)